In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

# ---------------------------------------------------------
# 1. STABLECOIN SUPPLY (USDT, USDC) – via CoinGecko
# ---------------------------------------------------------

def fetch_stablecoin_supply(coin_id="tether"):
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": "90"}
    r = requests.get(url, params=params)
    data = r.json()

    df = pd.DataFrame(data["market_caps"], columns=["timestamp", "market_cap"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["supply_est"] = df["market_cap"]  # proxy for supply
    return df[["date", "supply_est"]]

usdt = fetch_stablecoin_supply("tether")
usdc = fetch_stablecoin_supply("usd-coin")

# ---------------------------------------------------------
# 2. ETF FLOWS – via Farside CSV
# ---------------------------------------------------------

def fetch_etf_flows():
    url = "https://farside.co.uk/wp-content/uploads/bitcoin_etf_flow_table.csv"
    df = pd.read_csv(url)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.sort_values("Date")
    df["Net_Flow"] = df.iloc[:, 1:].sum(axis=1)  # sum across all ETF columns
    return df[["Date", "Net_Flow"]]

etf = fetch_etf_flows()

# ---------------------------------------------------------
# 3. MARKET DEPTH – via Binance Order Book Snapshot
# ---------------------------------------------------------

def fetch_market_depth(symbol="BTCUSDT", limit=1000):
    url = "https://api.binance.com/api/v3/depth"
    params = {"symbol": symbol, "limit": limit}
    r = requests.get(url, params=params).json()

    bids = pd.DataFrame(r["bids"], columns=["price", "qty"]).astype(float)
    asks = pd.DataFrame(r["asks"], columns=["price", "qty"]).astype(float)

    depth = {
        "bid_depth": (bids["price"] * bids["qty"]).sum(),
        "ask_depth": (asks["price"] * asks["qty"]).sum(),
        "timestamp": dt.datetime.utcnow()
    }
    return pd.DataFrame([depth])

depth = fetch_market_depth()

# ---------------------------------------------------------
# 4. VISUALISATIONS
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(usdt["date"], usdt["supply_est"], label="USDT Supply (est.)")
plt.plot(usdc["date"], usdc["supply_est"], label="USDC Supply (est.)")
plt.title("Stablecoin Supply (90 days)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.bar(etf["Date"], etf["Net_Flow"])
plt.title("Bitcoin ETF Net Flows")
plt.grid(True)
plt.show()

print("Latest Market Depth Snapshot:")
print(depth)

HTTPError: HTTP Error 403: Forbidden

In [4]:
def fetch_etf_flows():
    url = "https://api.bitmex.com/etf/flows"
    r = requests.get(url)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    df["net_flow"] = df["flow"]
    return df[["date", "net_flow"]]

etf = fetch_etf_flows()
etf.head()


ConnectionError: HTTPSConnectionPool(host='api.bitmex.com', port=443): Max retries exceeded with url: /etf/flows (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x00000262F8974D30>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed'))